# 📐 Constrained Optimization & Lagrange Multipliers

**Course**: Machine Learning Mathematics — Lecture Notes
**Topic**: Equality-Constrained Optimization, SciPy Solvers, and Method of Lagrange Multipliers

## 1. Constrained Optimization & Equality Constraints

### Intuition First
In real-world machine learning tasks, parameter optimization is often subject to physical or mathematical constraints—such as unit-norm weight constraints ($\|w\|_2 = 1$), budget bounds, or probability normalization ($\sum p_i = 1$). Constrained optimization seeks parameter choices that minimize objective loss while strictly satisfying all constraint equations.

### Formal Definition / Math
A single-variable equality-constrained optimization problem is formulated as:

$$\begin{equation}
\min_{x} f(x) \quad \text{subject to} \quad g(x) = 0
\end{equation}$$

#### Worked Example:
For $f(x) = x^2$ subject to $g(x) = x - 1 = 0 \implies x = 1$:
Direct substitution yields optimal solution $x^* = 1$ with minimum value $f(1) = 1.0$.

### Connection to the Code
- `from scipy.optimize import minimize`: Imports SciPy's numerical optimization solver.
- `res = minimize(f, [0], constraints={'type': 'eq', 'fun': g})`: Solves constrained problem using Sequential Least Squares Programming (SLSQP).

**Key Takeaway**: **SciPy's `minimize` solver finds parameter values minimizing $f(x)$ subject to equality constraints $g(x) = 0$.**

In [1]:
from scipy.optimize import minimize


def f(x):
    return x**2


def g(x):
    return x - 1

In [2]:
res = minimize(f, [0], constraints={"type": "eq", "fun": g})

print(res.x)
print(res.fun)

[1.]
1.0


## 2. Bivariate Constrained Optimization

### Intuition First
We extend constrained minimization to 2D spaces, finding the point on a constraint line $x + y = 1$ that is closest to the origin $(0, 0)$.

### Formal Definition / Math
The 2D constrained problem is:

$$\begin{equation}
\min_{x, y} f(x, y) = x^2 + y^2 \quad \text{subject to} \quad g(x, y) = x + y - 1 = 0
\end{equation}$$

By symmetry, $x = y = 0.5$, yielding minimum squared distance $f(0.5, 0.5) = 0.25 + 0.25 = 0.5$.

### Connection to the Code
- `res = minimize(f, [0, 0], constraints={'type': 'eq', 'fun': g})` returns optimal vector $[0.5, 0.5]^T$ and minimum value $0.5$.

**Key Takeaway**: **Bivariate constrained optimization solves multi-variable feature constraints such as $x + y = 1$.**

In [3]:
def f(A):
    x, y = A
    return x**2 + y**2


def g(A):
    x, y = A
    return x + y - 1


res = minimize(f, [0, 0], constraints={"type": "eq", "fun": g})

print(res.x)
print(res.fun)

[0.5 0.5]
0.49999999999999967


## 3. Multiple Simultaneous Equality Constraints

### Intuition First
Optimization often involves multiple simultaneous constraints, such as $x = -1$ and $y = -1$.

### Connection to the Code
- `cons = ({'type': 'eq', 'fun': g1}, {'type': 'eq', 'fun': g2})`: Passes a tuple of constraint dictionaries to `minimize`.

**Key Takeaway**: **Multiple simultaneous constraints are passed as tuples of constraint functions to numerical solvers.**

In [4]:
def f(A):
    x, y = A
    return x**2 + y**2


def g1(A):
    x, y = A
    return x + 1


def g2(A):
    x, y = A
    return y + 1


cons = ({"type": "eq", "fun": g1}, {"type": "eq", "fun": g2})
res = minimize(f, (0, 0), constraints=cons)

print(res.x)
print(res.fun)

[-1. -1.]
2.0


## 4. The Method of Lagrange Multipliers & Autograd Integration

### Intuition First
The **Method of Lagrange Multipliers** converts a constrained optimization problem into an unconstrained problem by introducing scalar multipliers $\lambda$. At an optimal constrained point, the gradient of the objective function $\nabla f$ is parallel to the gradient of the constraint function $\nabla g$.

### Formal Definition / Math
The Lagrangian function $\mathcal{L}(x, y, \lambda)$ is defined as:

$$\begin{equation}
\mathcal{L}(x, y, \lambda) = f(x, y) - \lambda \cdot g(x, y)
\end{equation}$$

Setting the gradient of the Lagrangian to zero yields the system of equations:

$$\begin{aligned}
\frac{\partial \mathcal{L}}{\partial x} &= \frac{\partial f}{\partial x} - \lambda \frac{\partial g}{\partial x} = 0 \\
\frac{\partial \mathcal{L}}{\partial y} &= \frac{\partial f}{\partial y} - \lambda \frac{\partial g}{\partial y} = 0 \\
\frac{\partial \mathcal{L}}{\partial \lambda} &= g(x, y) = 0
\end{aligned}$$

#### Worked Example:
For $f(x, y) = x + y$ subject to $g(x, y) = x^2 + y^2 - 1 = 0$ (unit circle constraint):
- $\mathcal{L}(x, y, \lambda) = x + y - \lambda(x^2 + y^2 - 1)$
- $\frac{\partial \mathcal{L}}{\partial x} = 1 - 2\lambda x = 0 \implies x = \frac{1}{2\lambda}$
- $\frac{\partial \mathcal{L}}{\partial y} = 1 - 2\lambda y = 0 \implies y = \frac{1}{2\lambda}$
- Substituting into $x^2 + y^2 = 1 \implies 2 \left(\frac{1}{2\lambda}\right)^2 = 1 \implies \lambda = \frac{1}{\sqrt{2}} \approx 0.7071$.
- Exact solution: $x^* = y^* = \frac{1}{\sqrt{2}} \approx 0.70710678$.

### Connection to the Code
- `from autograd import grad`: Uses automatic differentiation to compute analytical gradients of the Lagrangian.
- `fsolve(obj, [0.0, 0.0, 1.0])`: Uses SciPy's non-linear root finder to locate stationary points $[x^*, y^*, \lambda^*]^T$.

**Key Takeaway**: **Lagrange multipliers convert constrained problems into unconstrained system roots where $\nabla f = \lambda \nabla g$.**

In [5]:
try:
    from autograd import grad
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "autograd"])
    from autograd import grad
from scipy.optimize import fsolve


def f(A):
    x, y = A
    return x + y


def g(A):
    x, y = A
    return x**2 + y**2 - 1


def Lagrange(L):
    x, y, lam = L
    return f([x, y]) - lam * g([x, y])

In [6]:
gl = grad(Lagrange, 0)


def obj(L):
    x, y, l = L
    a, b, c = gl(L)
    return [a, b, g([x, y])]

In [7]:
x_opt, y_opt, l_opt = fsolve(obj, [0.0, 0.0, 1.0])
print(x_opt)
print(y_opt)
print(l_opt)

0.7071067811865656
0.7071067811865656
0.7071067811866819
